## `시도별 csv 3개 생성 하는 내용들입니다`

# 2단계: 시/도별 개선율 산정 + SAW 입력 데이터 생성

1단계 - 전국 도시대기 평균 기준 계절관리제 시행 이후 겨울철 PM10/PM2.5 농도가 감소했는지 확인
2딘계 - 감소폭을 **17개 시/도별로 분해**해서, 개선이 미흡한 지역을 찾고 SAW 우선순위 산정에 사용할 입력 데이터를 만든다.

**SAW 대안**: 17개 시/도

**SAW 기준**
- PM2.5 개선율
- PM10 개선율
- 시행 후 PM2.5 절대농도

**우선순위 방향**
- 개선율은 낮을수록 개선이 미흡하므로 우선순위가 높다.
- 시행 후 PM2.5 절대농도는 높을수록 현재 오염 부담이 크므로 우선순위가 높다.

## `0~2 사전작업. `
## 0.라이브러리 및 경로 설정

노트북 실행 위치가 프로젝트 루트일 수도 있고 `src` 폴더일 수도 있으므로, 두 가지 경로를 모두 후보로 둔다.

In [1]:
from pathlib import Path
import pandas as pd

# 실행 위치가 프로젝트 루트인 경우와 src 폴더인 경우를 모두 대응한다.
DATA_PATH_CANDIDATES = [
    Path("data/processed/airkorea_daily.parquet"),
    Path("../data/processed/airkorea_daily.parquet"),
]

OUTPUT_DIR_CANDIDATES = [
    Path("data/processed"),
    Path("../data/processed"),
]

BEFORE_LABEL = "시행 전 (2015~2018)"
AFTER_LABEL = "시행 후 (2019~2023)"
WINTER_MONTHS = [12, 1, 2, 3]

def resolve_existing_path(candidates: list[Path]) -> Path:
    """후보 경로 중 실제 존재하는 첫 번째 파일 경로를 반환한다."""
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {candidates}")

def resolve_output_dir(candidates: list[Path]) -> Path:
    """후보 경로 중 실제 존재하는 첫 번째 출력 폴더를 반환한다."""
    for path in candidates:
        if path.exists():
            return path
    candidates[0].mkdir(parents=True, exist_ok=True)
    return candidates[0]

DATA_PATH = resolve_existing_path(DATA_PATH_CANDIDATES)
OUTPUT_DIR = resolve_output_dir(OUTPUT_DIR_CANDIDATES)

print("입력 파일:", DATA_PATH)
print("출력 폴더:", OUTPUT_DIR)

입력 파일: ..\data\processed\airkorea_daily.parquet
출력 폴더: ..\data\processed


## 1. 전처리 완료 데이터 읽기

In [2]:
daily = pd.read_parquet(DATA_PATH)
daily["date"] = pd.to_datetime(daily["date"])

print("전체 데이터:", daily.shape)
print("컬럼:", list(daily.columns))
print()
print(daily["network"].value_counts(dropna=False))

전체 데이터: (1779743, 9)
컬럼: ['station_code', 'station_name', 'region', 'date', 'pm10', 'pm25', 'install_year', 'network', 'sido']

network
도시대기     1401587
도로변대기     157826
교외대기       81708
NaN        75187
항만         39836
국가배경       23599
Name: count, dtype: int64


## 2. 도시대기 측정소만 필터링

1단계와 동일하게 `network == "도시대기"`만 사용한다.

도로변대기는 자동차 배출 영향이 강하고 지역적으로 편중될 수 있으므로 제외.

`network`가 없는 측정소도 이 필터에서 함께 제외.

In [3]:
urban = daily[daily["network"] == "도시대기"].copy()

print("도시대기 데이터:", urban.shape)
print("시/도 수:", urban["sido"].nunique())
print("시/도 목록:", sorted(urban["sido"].dropna().unique()))
print("측정소 수:", urban["station_code"].nunique())
print("기간:", urban["date"].min(), "~", urban["date"].max())

도시대기 데이터: (1401587, 9)
시/도 수: 17
시/도 목록: ['강원', '경기', '경남', '경북', '광주', '대구', '대전', '부산', '서울', '세종', '울산', '인천', '전남', '전북', '제주', '충남', '충북']
측정소 수: 530
기간: 2015-01-01 00:00:00 ~ 2024-12-31 00:00:00


## =================================

## 3. 겨울철 정의 및 시행 전후 그룹 구성

겨울철은 `12월~다음해 3월`로 정의한다.

예를 들어 `2019 겨울`은 `2019-12 ~ 2020-03`이다. 따라서 1~3월은 전년도 겨울로 귀속한다.

- 시행 전: 2015~2018 겨울
- 시행 후: 2019~2023 겨울

```2014 겨울과 2024 겨울은 잘려서 제외함.```

In [4]:
winter = urban[urban["date"].dt.month.isin(WINTER_MONTHS)].copy()

############################## 12월은 그 해 겨울, 1~3월은 전년도 겨울로 묶음.
winter["winter_year"] = winter["date"].dt.year.where(
    winter["date"].dt.month == 12,
    winter["date"].dt.year - 1,
)

winter = winter[winter["winter_year"].between(2015, 2023)] ############################## 2014 겨울, 2024 겨울 제외함

# 계절관리제 최초 시행은 2019년 12월이므로 2019 겨울부터 시행 후로 본다.
winter["period"] = winter["winter_year"].apply(
    lambda year: BEFORE_LABEL if year <= 2018 else AFTER_LABEL ############################## 이제 2개 비교 하면 됨

)

print(winter.groupby(["winter_year", "period"]).size())

winter_year  period          
2015         시행 전 (2015~2018)    27385
2016         시행 전 (2015~2018)    29386
2017         시행 전 (2015~2018)    32493
2018         시행 전 (2015~2018)    41596
2019         시행 후 (2019~2023)    50286
2020         시행 후 (2019~2023)    56956
2021         시행 후 (2019~2023)    60148
2022         시행 후 (2019~2023)    62433
2023         시행 후 (2019~2023)    63624
dtype: int64


## 4. 시/도별 일평균으로 축약

원자료는 측정소-일 단위다. 시/도별 개선율을 계산하기 전에 `시/도 x 날짜 x 시행 전후` 단위로 평균을 낸다.

이렇게 하면 각 시/도는 하루에 PM10 하나, PM2.5 하나를 갖게 된다. 이후 시행 전후 평균은 이 일별 평균을 기준으로 계산한다.

In [5]:
sido_daily = (
    winter.groupby(["sido", "date", "period"])[["pm10", "pm25"]]
    .mean()
    .reset_index()
)

print("시/도별 일평균 데이터:", sido_daily.shape)
sido_daily.head()

시/도별 일평균 데이터: (18533, 5)


,sido,date,period,pm10,pm25
0,강원,2015-12-01,시행 전 (2015~2018),65.658333,43.041667
1,강원,2015-12-02,시행 전 (2015~2018),54.685606,45.000000
2,강원,2015-12-03,시행 전 (2015~2018),24.091667,14.875000
3,강원,2015-12-04,시행 전 (2015~2018),39.736667,18.145833
4,강원,2015-12-05,시행 전 (2015~2018),35.423188,17.562500


## 5. 시/도별 시행 전후 평균 및 개선율 계산

개선율 공식은 다음과 같다.

`개선율 = (시행 전 평균 - 시행 후 평균) / 시행 전 평균 * 100`

값이 클수록 시행 후 농도가 더 많이 감소했다는 뜻이다. 값이 낮으면 개선이 미흡한 지역으로 볼 수 있다.

In [6]:
period_mean = (
    sido_daily.groupby(["sido", "period"])[["pm10", "pm25"]]
    .mean()
    .unstack("period")
)

improvement = pd.DataFrame(
    {
        "pm25_before": period_mean[("pm25", BEFORE_LABEL)],
        "pm25_after": period_mean[("pm25", AFTER_LABEL)],
        "pm10_before": period_mean[("pm10", BEFORE_LABEL)],
        "pm10_after": period_mean[("pm10", AFTER_LABEL)],
    }
).reset_index()

improvement["pm25_improvement_rate"] = (
    (improvement["pm25_before"] - improvement["pm25_after"])
    / improvement["pm25_before"]
    * 100
)
improvement["pm10_improvement_rate"] = (
    (improvement["pm10_before"] - improvement["pm10_after"])
    / improvement["pm10_before"]
    * 100
)

# PM2.5 개선율이 낮은 지역부터 확인한다.
improvement = improvement.sort_values("pm25_improvement_rate").reset_index(drop=True)

improvement.round(2)

,sido,pm25_before,pm25_after,pm10_before,pm10_after,pm25_improvement_rate,pm10_improvement_rate
0,충남,30.44,27.62,51.51,47.35,9.25,8.06
1,세종,29.82,26.07,52.31,46.28,12.55,11.54
2,인천,30.95,25.19,52.57,44.06,18.61,16.18
3,서울,31.95,25.61,53.53,44.03,19.83,17.75
4,대구,29.29,22.93,48.71,40.18,21.72,17.52
5,경기,35.43,27.28,60.10,47.28,23.02,21.33
6,대전,29.43,22.45,53.72,42.00,23.73,21.82
7,광주,29.94,22.25,47.82,37.91,25.69,20.71
8,충북,36.81,27.26,55.17,44.88,25.94,18.65
9,울산,25.32,18.64,43.11,34.86,26.38,19.14


## 6. 개선 미흡 지역 확인

PM2.5 개선율 기준으로 하위 지역을 먼저 확인한다. 이 지역들은 SAW에서 높은 우선순위를 받을 가능성이 크다.

In [7]:
improvement.head(5).round(2)

,sido,pm25_before,pm25_after,pm10_before,pm10_after,pm25_improvement_rate,pm10_improvement_rate
0,충남,30.44,27.62,51.51,47.35,9.25,8.06
1,세종,29.82,26.07,52.31,46.28,12.55,11.54
2,인천,30.95,25.19,52.57,44.06,18.61,16.18
3,서울,31.95,25.61,53.53,44.03,19.83,17.75
4,대구,29.29,22.93,48.71,40.18,21.72,17.52


## 7. 편향 점검표 생성

개선율만 보고 결론을 내리면 위험할 수 있다.

시/도별 측정소 수, 관측 행 수, 결측률이 다를 수 있기 때문이다. 특히 PM2.5는 과거 기간에 결측이 상대적으로 많을 수 있다.

따라서 보고서에는 개선율 결과와 함께 편향 가능성을 정리해야 한다.

In [8]:
bias = (
    winter.groupby(["sido", "period"])
    .agg(
        station_count=("station_code", "nunique"),
        row_count=("station_code", "size"),
        pm25_missing_rate=("pm25", lambda s: s.isna().mean() * 100),
        pm10_missing_rate=("pm10", lambda s: s.isna().mean() * 100),
    )
    .reset_index()
)

bias.round(2)

,sido,period,station_count,row_count,pm25_missing_rate,pm10_missing_rate
0,강원,시행 전 (2015~2018),19,3900,9.18,2.15
1,강원,시행 후 (2019~2023),24,13506,5.91,2.84
2,경기,시행 전 (2015~2018),79,33230,34.94,4.68
3,경기,시행 후 (2019~2023),109,62334,3.57,3.40
4,경남,시행 전 (2015~2018),26,9543,26.61,4.36
5,경남,시행 후 (2019~2023),39,22298,2.99,2.79
6,경북,시행 전 (2015~2018),20,7243,50.68,9.00
7,경북,시행 후 (2019~2023),49,25720,6.11,3.86
8,광주,시행 전 (2015~2018),6,2910,9.90,1.13
9,광주,시행 후 (2019~2023),11,6311,2.44,1.85


## 8. SAW 입력 데이터 생성

SAW 기준은 다음 세 가지다.

1. PM2.5 개선율
2. PM10 개선율
3. 시행 후 PM2.5 절대농도

개선 우선순위 관점에서는 정규화 방향이 중요하다.

- PM2.5 개선율: 낮을수록 우선순위 높음 → cost 기준
- PM10 개선율: 낮을수록 우선순위 높음 → cost 기준
- 시행 후 PM2.5 절대농도: 높을수록 우선순위 높음 → benefit 기준

In [9]:
def normalize_benefit(series: pd.Series) -> pd.Series:
    """값이 클수록 좋은 기준을 0~1로 정규화한다."""
    min_value = series.min()
    max_value = series.max()
    return (series - min_value) / (max_value - min_value)


def normalize_cost(series: pd.Series) -> pd.Series:
    """값이 작을수록 좋은 기준을 0~1로 정규화한다."""
    min_value = series.min()
    max_value = series.max()
    return (max_value - series) / (max_value - min_value)


saw = improvement[
    [
        "sido",
        "pm25_improvement_rate",
        "pm10_improvement_rate",
        "pm25_after",
    ]
].copy()

saw = saw.rename(columns={"pm25_after": "pm25_after_abs"})

# 개선율은 낮을수록 우선순위가 높으므로 cost 기준으로 정규화한다.
saw["pm25_improvement_priority"] = normalize_cost(saw["pm25_improvement_rate"])
saw["pm10_improvement_priority"] = normalize_cost(saw["pm10_improvement_rate"])

# 시행 후 PM2.5 절대농도는 높을수록 우선순위가 높으므로 benefit 기준으로 정규화한다.
saw["pm25_after_abs_priority"] = normalize_benefit(saw["pm25_after_abs"])

saw.round(3)

,sido,pm25_improvement_rate,pm10_improvement_rate,pm25_after_abs,pm25_improvement_priority,pm10_improvement_priority,pm25_after_abs_priority
0,충남,9.250,8.065,27.620,1.000,1.000,1.000
1,세종,12.552,11.535,26.073,0.877,0.840,0.861
2,인천,18.613,16.178,25.187,0.650,0.627,0.781
3,서울,19.829,17.754,25.613,0.605,0.554,0.819
4,대구,21.723,17.517,22.930,0.534,0.565,0.577
5,경기,23.017,21.333,27.278,0.486,0.389,0.969
6,대전,23.732,21.816,22.447,0.459,0.367,0.534
7,광주,25.687,20.711,22.246,0.386,0.418,0.516
8,충북,25.943,18.651,27.257,0.377,0.513,0.967
9,울산,26.382,19.141,18.637,0.360,0.490,0.191


## 9. 동일가중 SAW 점수 및 순위 계산

우선은 세 기준을 동일가중치로 계산한다.

이후 PM2.5를 더 중요하게 볼지, 절대농도를 더 중요하게 볼지에 따라 가중치를 바꿔 민감도 분석을 할 수 있다.

In [10]:
weights = {
    "pm25_improvement_priority": 1 / 3,
    "pm10_improvement_priority": 1 / 3,
    "pm25_after_abs_priority": 1 / 3,
}

saw["saw_score_equal_weight"] = sum(
    saw[column] * weight for column, weight in weights.items()
)

saw["saw_rank_equal_weight"] = (
    saw["saw_score_equal_weight"].rank(ascending=False, method="min").astype(int)
)

saw = saw.sort_values("saw_rank_equal_weight").reset_index(drop=True)

saw.round(3)

,sido,pm25_improvement_rate,pm10_improvement_rate,pm25_after_abs,pm25_improvement_priority,pm10_improvement_priority,pm25_after_abs_priority,saw_score_equal_weight,saw_rank_equal_weight
0,충남,9.250,8.065,27.620,1.000,1.000,1.000,1.000,1
1,세종,12.552,11.535,26.073,0.877,0.840,0.861,0.859,2
2,인천,18.613,16.178,25.187,0.650,0.627,0.781,0.686,3
3,서울,19.829,17.754,25.613,0.605,0.554,0.819,0.659,4
4,충북,25.943,18.651,27.257,0.377,0.513,0.967,0.619,5
5,경기,23.017,21.333,27.278,0.486,0.389,0.969,0.615,6
6,대구,21.723,17.517,22.930,0.534,0.565,0.577,0.559,7
7,대전,23.732,21.816,22.447,0.459,0.367,0.534,0.453,8
8,광주,25.687,20.711,22.246,0.386,0.418,0.516,0.440,9
9,전남,27.613,11.394,17.829,0.314,0.847,0.118,0.426,10


## 10. 결과 저장

아래 세 파일을 저장한다.

- `sido_winter_improvement.csv`: 시/도별 시행 전후 평균 및 개선율
- `sido_winter_bias_check.csv`: 측정소 수, 행 수, 결측률 점검표
- `sido_saw_input_equal_weight.csv`: SAW 입력값, 정규화 점수, 동일가중 순위

In [11]:
improvement_path = OUTPUT_DIR / "sido_winter_improvement.csv"
bias_path = OUTPUT_DIR / "sido_winter_bias_check.csv"
saw_path = OUTPUT_DIR / "sido_saw_input_equal_weight.csv"

# Excel에서 한글이 깨지지 않도록 utf-8-sig로 저장한다.
improvement.to_csv(improvement_path, index=False, encoding="utf-8-sig")
bias.to_csv(bias_path, index=False, encoding="utf-8-sig")
saw.to_csv(saw_path, index=False, encoding="utf-8-sig")

print("개선율 저장:", improvement_path)
print("편향 점검 저장:", bias_path)
print("SAW 입력 저장:", saw_path)

개선율 저장: ..\data\processed\sido_winter_improvement.csv
편향 점검 저장: ..\data\processed\sido_winter_bias_check.csv
SAW 입력 저장: ..\data\processed\sido_saw_input_equal_weight.csv


## 11. 해석 메모

- PM2.5 개선율이 낮고, 시행 후 PM2.5 절대농도가 높은 지역은 개선 우선순위가 높게 나온다.
- 단, 이 결과는 전후 비교 기반이다. 코로나19, 중국발 오염 변화, 산업 구조 변화, 측정망 변화 등 외부 요인을 분리한 인과효과는 아니다.
- 시/도별 측정소 수와 결측률 차이가 있으므로 `bias` 표를 함께 확인해야 한다.
- SAW 가중치는 정책 판단에 따라 바뀔 수 있으므로 동일가중 결과만 최종 결론으로 고정하지 않는 것이 좋다.